# Notebook 08: Project Integration

**Time:** 35 minutes  
**Prerequisites:** All previous notebooks (00–07) complete  
**Goal:** Connect everything from Week 2 to your capstone project

By the end of this notebook, you will have:
1. Reviewed your Week 1 project definition
2. Designed a data strategy for your project using Week 2 skills
3. Run your project text through the full pipeline (collect → clean → tokenize)
4. Considered architecture constraints for your use case
5. Produced `outputs/my_project_update.md` with your updated plan

In [1]:
import os, sys, json, time
from pathlib import Path
from datetime import datetime

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'))

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection, estimate_tokens
from src.tokenizer_utils import estimate_tokens_tiktoken
from src.data_utils import run_cleaning_pipeline, detect_languages
import src.config as config

client  = LLMClient(path=config.PATH)
tracker = CostTracker()

outputs_dir = os.path.join('..', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

print("✅ Setup complete — ready for Notebook 08")

/Users/scottlai/Documents/inferenceAI/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✓ Claude API client initialized
  Default model: claude-sonnet-4-6
  Available: claude-sonnet-4-6, claude-opus-4-6, claude-haiku-4-5-20251001
✅ Setup complete — ready for Notebook 08


---

## Part 1: Review Your Week 1 Project Definition

Let's load your project definition from Homework 1 (if available) and see how Week 2 knowledge changes your perspective.

In [2]:
print("=" * 65)
print("📋 Part 1: Your Week 1 Project Definition")
print("=" * 65)
print()

# Try to load from HW1 outputs
hw1_project_path = os.path.join(parent_dir, '..', 'Homework1-Submission', 'outputs', 'my_project_definition.md')
hw1_text = ""

if os.path.exists(hw1_project_path):
    with open(hw1_project_path) as f:
        hw1_text = f.read()
    print("✅ Found Week 1 project definition:")
    print()
    print(hw1_text[:1000])
    if len(hw1_text) > 1000:
        print(f"\n... ({len(hw1_text) - 1000} more chars)")
else:
    print("⚠️  Week 1 project definition not found.")
    print("   No worries — fill in the template below.")
    print()
    hw1_text = """
# My Project Definition

**Project Title:** [Your project title]
**Domain:** [e.g., medical QA, code review, financial analysis, etc.]
**Problem:** [What problem does your project solve?]
**Approach:** [High-level technical approach]
**Success Criteria:** [How will you know it works?]
"""
    print(hw1_text)
    print("💡 Fill in the template above in a text editor or update the variables below.")

📋 Part 1: Your Week 1 Project Definition

✅ Found Week 1 project definition:

# My Research Agent Project
**Created:** 2026-03-21 23:43:09


# MY RESEARCH AGENT PROJECT

## 1. PROJECT TITLE
[Your project name]

## 2. THE PROBLEM
[What research problem are you solving?]

## 3. YOUR SOLUTION
[How will your agent solve this?]

## 4. USER WORKFLOW
[How will someone use it?]

## 5. COMPONENTS
[Which course techniques will you use?]

☐ CO-STAR prompting - 
☐ Structured outputs - 
☐ Chain-of-thought - 
☐ Model selection - 
☐ MCP/Tool use - 
☐ Multi-step workflow - 

## 6. SUCCESS CRITERIA
[How will you measure success?]

## 7. SCOPE
IN SCOPE:
- 
- 
- 

OUT OF SCOPE:
- 
- 

## 8. DATA SOURCES
[Where does your data come from?]

## 9. TECH STACK
[What tools/libraries?]

## 10. TIMELINE
Week 1:
- 

Week 2-3:
- 

Week 4 (Insight I):
- 

Week 5-6:
- 

Week 7 (Insight II):
- 

Week 8-10:
- 

## 11. RISKS & MITIGATION
Risk 1: 
  Mitigation: 

Risk 2: 
  Mitigation: 

## 12. STRETCH GOALS
- 
- 


---


---

## Part 2: Data Strategy for Your Project

In [3]:
print("=" * 65)
print("🧪 Experiment 1: Claude Designs Your Data Strategy")
print("=" * 65)
print()

data_strategy_prompt = f"""
Based on this project definition:

{hw1_text[:2000]}

Design a data collection and preparation strategy for this project.
Cover:
1. What data sources would you scrape/collect? (web, PDFs, audio, databases?)
2. Expected data volume (documents, tokens)
3. Key data cleaning concerns (PII? duplicates? language? quality?)
4. Tokenization strategy (which tokenizer, estimated cost per inference)
5. Would this project benefit from fine-tuning, RAG, or prompt engineering alone?

Be practical and specific. Assume the student has access to the tools we've covered:
trafilatura, pytesseract, faster-whisper, datasketch, presidio, tiktoken.
"""

resp_strategy = client.generate(
    prompt=data_strategy_prompt,
    system="You are a senior ML engineer advising a student on their capstone project data strategy.",
    max_tokens=800,
    temperature=0.5
)

if "error" not in resp_strategy:
    tracker.add_call(resp_strategy)
    print(format_response(resp_strategy, verbose=True))
else:
    print(f"❌ {resp_strategy['error']}")

🧪 Experiment 1: Claude Designs Your Data Strategy

Model: claude-sonnet-4-6
Tokens: 844 in, 800 out
Stop reason: max_tokens
# Data Collection & Preparation Strategy

## ⚠️ Important Caveat First

Your project template is **completely blank**, which means I'm working without knowing:
- What domain you're researching
- What problem you're solving
- Who your users are

**I'll build a practical framework you can adapt**, using a concrete example assumption: *a research agent that helps graduate students synthesize academic literature on a given topic*. Adjust every number and decision below to match your actual project.

---

## 1. Data Sources to Collect

### Primary Sources (Ranked by Reliability)

```
┌─────────────────────┬──────────────┬─────────────┬─────────────────────┐
│ Source              │ Tool         │ Format      │ Reliability         │
├─────────────────────┼──────────────┼─────────────┼─────────────────────┤
│ arXiv / PubMed      │ API (free)   │ PDF + XML   │ ⭐⭐⭐⭐⭐ Best s

### 🎯 TODO 1: Fill In Your Data Strategy

In [4]:
# TODO 1: Fill in your project's data strategy based on Claude's advice
#         and your own domain knowledge

data_strategy = {
    "project_title":       "[YOUR PROJECT TITLE]",
    "domain":              "[YOUR DOMAIN]",
    
    # Data collection
    "scraping_targets":    [
        # "arXiv cs.CL papers",
        # "Company documentation PDFs",
        # "YouTube lecture transcripts",
    ],
    "expected_volume":     "[e.g., 500 documents, ~200K tokens]",
    
    # Data cleaning
    "cleaning_concerns":   [
        # "PII in customer emails",
        # "Duplicate abstracts from cross-posted papers",
        # "Non-English content to filter",
    ],
    
    # Tokenization
    "tokenizer":           "[tiktoken cl100k_base / HF gpt2 / sentencepiece]",
    "est_cost_per_call":   "[$ per API call]",
    
    # Approach
    "approach":            "[fine-tuning / RAG / prompt engineering / hybrid]",
    "approach_rationale":  "[Why this approach for your use case?]",
}

print("=" * 65)
print("🎯 TODO 1: My Data Strategy")
print("=" * 65)
print()

for key, val in data_strategy.items():
    print(f"  {key:<25} {val}")

🎯 TODO 1: My Data Strategy

  project_title             [YOUR PROJECT TITLE]
  domain                    [YOUR DOMAIN]
  scraping_targets          []
  expected_volume           [e.g., 500 documents, ~200K tokens]
  cleaning_concerns         []
  tokenizer                 [tiktoken cl100k_base / HF gpt2 / sentencepiece]
  est_cost_per_call         [$ per API call]
  approach                  [fine-tuning / RAG / prompt engineering / hybrid]
  approach_rationale        [Why this approach for your use case?]


---

## Part 3: Mini Pipeline Run

Run your project-specific data through the complete pipeline we built in notebooks 03–05.

In [5]:
# TODO 2: Provide project-specific text samples and run the full pipeline

# Option A: Use your arXiv scraping results from NB04
# Option B: Paste in domain-specific text manually
# Option C: Generate synthetic domain text with Claude

print("=" * 65)
print("🎯 TODO 2: Mini Pipeline Run")
print("=" * 65)
print()

# Try loading from NB04 results
arxiv_path = os.path.join(outputs_dir, 'arxiv_clean.json')
project_texts = []

if os.path.exists(arxiv_path):
    with open(arxiv_path) as f:
        papers = json.load(f)
    project_texts = [p.get('raw_text', p.get('abstract', '')) for p in papers if p.get('raw_text') or p.get('abstract')]
    print(f"✅ Loaded {len(project_texts)} texts from NB04 arXiv results")

if not project_texts:
    print("⚠️  No NB04 results found — generating synthetic data")
    resp_synth = client.generate(
        prompt=(
            f"Generate 5 short domain-specific text samples (each 50-100 words) "
            f"for the field of: {data_strategy.get('domain', 'machine learning')}. "
            f"Mix technical papers, web content, and conversational text."
        ),
        max_tokens=800, temperature=0.8
    )
    if "error" not in resp_synth:
        tracker.add_call(resp_synth)
        project_texts = [p.strip() for p in resp_synth['content'].split('\n\n') if len(p.strip()) > 50]
        print(f"  Generated {len(project_texts)} synthetic samples")

if project_texts:
    # Step 1: Tokenize
    print("\n--- Tokenization ---")
    total_tokens = sum(estimate_tokens_tiktoken(t) for t in project_texts)
    print(f"  Total tokens (tiktoken): {total_tokens:,}")

    # Step 2: Clean
    print("\n--- Cleaning Pipeline ---")
    result = run_cleaning_pipeline(
        raw_texts=project_texts,
        lang_filter='en',
        dedup_threshold=0.7,
        output_dir=outputs_dir
    )

    clean_texts = result['clean_texts']
    clean_tokens = sum(estimate_tokens_tiktoken(t) for t in clean_texts)
    print(f"\nAfter cleaning: {len(clean_texts)} docs, {clean_tokens:,} tokens")
    print(f"Retention rate: {len(clean_texts)/len(project_texts)*100:.0f}% of documents")

    # Save a small sample
    sample_path = os.path.join(outputs_dir, 'project_corpus_sample.txt')
    with open(sample_path, 'w') as f:
        f.write('\n\n---\n\n'.join(clean_texts[:10]))
    print(f"✅ Sample saved: {sample_path}")

🎯 TODO 2: Mini Pipeline Run

✅ Loaded 10 texts from NB04 arXiv results

--- Tokenization ---
  Total tokens (tiktoken): 4,044

--- Cleaning Pipeline ---

📊 Cleaning pipeline — 10 input documents

[Stage 1] HTML stripping & whitespace normalisation...
  → 10 docs remaining

[Stage 2] Language filtering (keeping: 'en')...
  → 10 docs remaining

[Stage 3] MinHash deduplication (threshold=0.7)...
🔁 MINHASH DEDUPLICATION RESULTS
  Input documents:    10
  Duplicates removed: 0
  Output documents:   10
  Removal rate:       0.0%
  Threshold:          Jaccard ≥ 0.7
  → 10 docs remaining

[Stage 4] PII removal...
  → 32 PII entities anonymised

✓ Clean corpus saved: ../outputs/clean_corpus.txt
✓ Stats saved:        ../outputs/corpus_stats.md

✅ PIPELINE COMPLETE
  0_original                         10 docs
  1_after_html_strip                 10 docs
  2_after_lang_filter                10 docs
  3_after_dedup                      10 docs
  4_after_pii                        10 docs
  Estimate

---

## Part 4: Architecture Reflection

In [6]:
# TODO 3: Ask Claude how Transformer architecture constraints affect your project

arch_prompt = f"""
Given that Transformers use self-attention with O(n²) complexity in the sequence length:

For my project in the domain of '{data_strategy.get('domain', 'machine learning')}':

1. What typical input lengths will I encounter?
2. Will any inputs exceed a 128K context window? If so, what strategies should I use?
   (chunking, RAG, summarization, hierarchical approaches)
3. For my cost budget, should I use extended thinking or save tokens by using CoT prompting?
4. If I later want to fine-tune, should I start with a smaller model (7B) or larger (27B+)?
   Consider my available hardware and use case.

Be specific to my domain — not generic advice.
"""

print("=" * 65)
print("🎯 TODO 3: Architecture Constraints for My Project")
print("=" * 65)
print()

resp_arch = client.generate(
    prompt=arch_prompt,
    system="You are a systems architect with deep knowledge of LLM deployment constraints.",
    max_tokens=600,
    temperature=0.3
)

if "error" not in resp_arch:
    tracker.add_call(resp_arch)
    print(format_response(resp_arch, verbose=True))

todo3_reflection = """
[YOUR REFLECTION HERE]

- What surprised you about the architecture constraints for your domain?
- How will context window limits affect your project design?
- What trade-off between model size and latency makes sense for your use case?
"""
print()
print(todo3_reflection)

🎯 TODO 3: Architecture Constraints for My Project

Model: claude-sonnet-4-6
Tokens: 184 in, 329 out
Stop reason: end_turn
I notice your message contains a placeholder '[YOUR DOMAIN]' that wasn't filled in. I can't give you domain-specific advice without knowing what you're actually building.

**To get a useful answer, tell me:**

- What is your actual domain? (e.g., legal document review, genomics, customer support, code generation, medical imaging reports, financial analysis)
- What kind of inputs does your system process? (documents, conversations, structured data, code, etc.)
- What is your approximate cost budget or infrastructure constraints?
- What hardware do you have available for potential fine-tuning?

---

**Why this matters concretely:**

The answers diverge dramatically by domain. For example:

- **Legal contracts** → inputs routinely hit 50K-200K tokens, RAG is often wrong choice, hierarchical summarization matters
- **Customer support chat** → inputs rarely exceed 2K tok

---

## Part 5: Generate Updated Project Definition

In [7]:
print("=" * 65)
print("📝 Generating my_project_update.md")
print("=" * 65)
print()

# Combine all information into the project update
project_update = f"""# Project Update — Week 2

**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
**Student Name:** [Your Name]

---

## Original Project Definition (Week 1)

{hw1_text[:1500] if hw1_text else '[Not available — fill in below]'}

---

## Week 2 Updates

### Data Strategy

| Aspect | Plan |
|---|---|
| Scraping targets | {data_strategy.get('scraping_targets', '[]')} |
| Expected volume | {data_strategy.get('expected_volume', 'TBD')} |
| Cleaning concerns | {data_strategy.get('cleaning_concerns', '[]')} |
| Tokenizer | {data_strategy.get('tokenizer', 'TBD')} |
| Est. cost per call | {data_strategy.get('est_cost_per_call', 'TBD')} |
| Approach | {data_strategy.get('approach', 'TBD')} |

### Architecture Constraints

{resp_arch.get('content', '[Run TODO 3 above to generate this section]')[:800] if 'resp_arch' in dir() and isinstance(resp_arch, dict) else '[Not yet generated]'}

### Key Learnings from Week 2

- Transformer attention is O(n²) — must plan for context window limits
- Tokenizer choice affects cost: tiktoken (cl100k) is most efficient for English
- Data cleaning pipeline: language filter → dedup → PII removal
- Extended thinking improves reasoning at ~2× token cost
- Fine-tuning covered conceptually; full implementation in later class

### Updated Technical Approach

[TODO: Based on what you learned in Week 2, update your approach here.
Consider: model choice, data pipeline, tokenization, context window strategy,
and whether fine-tuning or RAG makes more sense for your use case.]

---

*Updated after completing Week 2 notebooks 00–08.*
"""

update_path = os.path.join(outputs_dir, 'my_project_update.md')
with open(update_path, 'w') as f:
    f.write(project_update)

print(f"✅ Saved: {update_path}")
print()
print("📌 IMPORTANT: Open this file and fill in the [TODO] sections before submitting!")

📝 Generating my_project_update.md

✅ Saved: ../outputs/my_project_update.md

📌 IMPORTANT: Open this file and fill in the [TODO] sections before submitting!


## Final Cost Report & Reflection

In [8]:
print("=" * 65)
print("💰 FULL HOMEWORK 2 COST REPORT")
print("=" * 65)
print()
tracker.report(detailed=True)

# Final reflection combining all sections
final_reflection = f"""
### Project Data Strategy Summary

Domain: {data_strategy.get('domain', '[not set]')}
Approach: {data_strategy.get('approach', '[not set]')}
Rationale: {data_strategy.get('approach_rationale', '[not set]')}

### Architecture Constraints (TODO 3)

{todo3_reflection.strip() if 'todo3_reflection' in dir() else '[Not filled in]'}

### Week 2 Impact on My Project

[Summarize how Week 2 changed your thinking about your project:
 - What will you do differently based on understanding transformer architecture?
 - How does the data pipeline apply to your use case?
 - Which Week 2 tool was most useful for your project? Why?]

### Total API Cost for Homework 2

Calls: {tracker.get_summary()['total_calls']}
Input tokens: {tracker.get_summary()['total_input_tokens']:,}
Output tokens: {tracker.get_summary()['total_output_tokens']:,}
Total cost: ${tracker.get_summary()['total_cost']:.4f}
"""

reflection_file = append_to_reflection(
    notebook="08",
    section_title="Project Integration & Week 2 Wrap-up",
    reflection_content=final_reflection,
    output_dir=os.path.join('..', 'outputs')
)
print(f"\n✅ Final reflection saved: {reflection_file}")

💰 FULL HOMEWORK 2 COST REPORT

💰 API COST REPORT
Total API calls:     2
Total input tokens:  1,028
Total output tokens: 1,129
Total cost:          $0.0200

All calls:
  1. [23:31:01] sonnet — 844in/800out — $0.0145
  2. [23:31:26] sonnet — 184in/329out — $0.0055

✅ Final reflection saved: ../outputs/homework_reflection.md


## ✅ Homework 2 Complete!

### Submission Checklist

Please verify these files exist in your `outputs/` directory:

- ✅ `homework_reflection.md` — **Primary deliverable** (70% of grade)
- ✅ `my_project_update.md` — **Secondary deliverable** (20% of grade)
- ✅ All notebooks executed with TODOs filled in (10% of grade)

### Supporting Output Files

- `path_selection.md`
- `setup_summary.txt`
- `tokenizer_comparison.json`
- `attention_heatmaps/` (PNG files)
- `arxiv_clean.json`
- `pdf_ocr/` (TXT files)
- `talks_transcripts.jsonl`
- `clean_corpus.txt` + `corpus_stats.md`
- `thinking_traces/` (JSON files)

### What You Learned in Week 2

1. **Transformer Architecture** — attention, multi-head, positional encoding, residual + LayerNorm
2. **Tokenization** — BPE, tiktoken vs HuggingFace vs SentencePiece, cost implications
3. **Pretraining Data Pipeline** — web scraping, PDF OCR, ASR transcription
4. **Data Cleaning** — language filtering, MinHash dedup, PII removal
5. **Fine-tuning & Alignment** — SFT, LoRA, DPO vs PPO (concepts)
6. **Test-Time Scaling** — CoT, extended thinking, quantization

### Next Week Preview

**Week 3** will dive into hands-on fine-tuning with PEFT/LoRA — taking the concepts from notebook 06 and turning them into working code!

---

**Congratulations on completing Week 2!** 🎉